In [1]:
import pandas as pd
import hvplot.pandas
import panel as pn
pn.extension()

In [2]:
df = pd.read_csv('zomato.csv',encoding='latin1')
df = df.dropna()


In [3]:

df.columns = df.columns.str.strip()

df = df[['City', 'Cuisines', 'Votes', 'Aggregate rating', 'Price range']]

df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce')
df['Aggregate rating'] = pd.to_numeric(df['Aggregate rating'], errors='coerce')

df = df.dropna()

In [4]:
kpi1 = pn.pane.Markdown(f"### 🍽️ Total Restaurants\n# {len(df)}")
kpi2 = pn.pane.Markdown(f"### ⭐ Avg Rating\n# {round(df['Aggregate rating'].mean(),2)}")
kpi3 = pn.pane.Markdown(f"### 👍 Total Votes\n# {int(df['Votes'].sum())}")
kpi4 = pn.pane.Markdown(f"### 🏙️ Cities\n# {df['City'].nunique()}")
kpi5 = pn.pane.Markdown(f"### 🍜 Cuisines\n# {df['Cuisines'].nunique()}")

kpis = pn.Row(kpi1, kpi2, kpi3, kpi4, kpi5)

In [5]:
bar = df.groupby('Cuisines')['Votes'] \
    .mean().nlargest(5).reset_index() \
    .hvplot.barh(x='Cuisines', y='Votes', title="Top Cuisines by Votes",
)
bar

:Bars   [Cuisines]   (Votes)

In [6]:
# area_line = (
#     df.groupby('Price range')['Aggregate rating']
#     .mean().reset_index()
#     .hvplot.area(x='Price range', y='Aggregate rating', alpha=0.4)
#     *
#     df.groupby('Price range')['Aggregate rating']
#     .mean().reset_index()
#     .hvplot.line(x='Price range', y='Aggregate rating')
# )
# area_line
ridge = df.hvplot.kde(
    y='Aggregate rating',
    by='Price range',
    height=250,
    width=500,
    alpha=0.6
)
ridge

:NdOverlay   [Price range]
   :Distribution   [Aggregate rating]   (Density)

In [7]:
hexbin = df.hvplot.hexbin(x='Votes', y='Aggregate rating', gridsize=20, height=300)
hexbin

:HexTiles   [Votes,Aggregate rating]

In [8]:
violin = df.hvplot.violin(y='Votes', by='Price range', height=300)
violin

:Violin   [Price range]   (Votes)

In [9]:
top_cities = df.groupby('City')['Votes'] \
    .sum().nlargest(10).index

heatmap = df[df['City'].isin(top_cities)] \
    .groupby(['Price range','City'])['Aggregate rating'] \
    .mean().reset_index() \
    .hvplot.heatmap(
        x='Price range',
        y='City',
        C='Aggregate rating',
        cmap='viridis',        # ✅ color added
        height=320,
        responsive=True,
        title="Top 10 Cities Heatmap"
    )
heatmap

:HeatMap   [Price range,City]   (Aggregate rating)

In [10]:
top_cities = df.groupby('City')['Votes'] \
    .sum().nlargest(10).index

scatter = df[df['City'].isin(top_cities)] \
    .groupby('City')[['Votes', 'Aggregate rating']] \
    .mean().reset_index() \
    .hvplot.scatter(
        x='Votes',
        y='Aggregate rating',
        by='City',
        height=320,
        size=120,
        responsive=True,
        title="Top 10 Cities: Votes vs Rating"
    )
scatter

:NdOverlay   [City]
   :Scatter   [Votes]   (Aggregate rating)

In [11]:
top_cities = df.groupby('City')['Votes'] \
    .sum().nlargest(10).index

top_cuisines = df.groupby('Cuisines')['Votes'] \
    .sum().nlargest(10).index

box = df[
    (df['City'].isin(top_cities)) &
    (df['Cuisines'].isin(top_cuisines))
].hvplot.box(
    y='Aggregate rating',
    by='City',
    groupby='Cuisines',
    height=320,
    responsive=True,
    title="Rating Distribution (Top Cities & Cuisines)"
)
box

:DynamicMap   [Cuisines]
   :BoxWhisker   [City]   (Aggregate rating)

In [12]:
insight1 = "### 📊 Insight 1\nHigher price ranges tend to have slightly better ratings."

insight2 = "### 📈 Insight 2\nRestaurants with more votes generally have higher ratings."

insight3 = "### 🍜 Insight 3\nFew cuisines dominate customer engagement and votes."

In [15]:


layout = pn.Column(

    pn.Column(
        pn.Row(
            bar.opts(height=300, width=1000),
            ridge.opts(height=300, width=500)
        )
    ),

    pn.Column(
        pn.Row(
            hexbin.opts(height=300, width=500),
            heatmap.opts(height=300, width=500)
        )
    ),

    pn.Column(
        pn.Row(
            scatter.opts(height=300, width=1020)
        )
    ),

    pn.Column(
        pn.Row(
            violin.opts(height=300, width=500),
            box.opts(height=300, width=500)
        )
    )
)



In [16]:

dashboard = pn.template.FastListTemplate(

    title="📊Zomato Dashboard",

    header=[],

    sidebar=[
        "## 📌 Insights",
        insight1,
        insight2,
        insight3
    ],

    main=[

        pn.Column(

            pn.Column(
                "## 📊 Key Metrics",

                pn.Row(
                    pn.panel(kpi1, width=250),
                    pn.panel(kpi2, width=250),
                    pn.panel(kpi3, width=250),
                    pn.panel(kpi4, width=250),
                    sizing_mode='fixed'
                ),

                margin=(0, 0, 20, 0)   
            ),

            layout
        )
    ]
)

dashboard.show()

Launching server at http://localhost:63211
